# Notebook v3 release — Full data / 12 methods

เลือก GPU บน Colab แล้วรันตามลำดับ อัปโหลด `Notebook_3_release_colab.zip` (โค้ดเท่านั้น)
ไป `MyDrive/nih -xray/` ข้อมูลอ่านจาก `MyDrive/nih -xray/data/3` ที่มีอยู่ ไม่อัปโหลดรูปซ้ำ

Train 46,108 / Validation 11,719 / Test 12,081; XAI123ภาพ ทุก method ใช้ manifests เดียวกัน
ค่าเริ่มต้น3seeds/20epochs รวม36CNN+3DAE; ไม่ใช่ notebook ที่รันเต็มได้ในไม่กี่นาที
Results/checkpoints ลง Drive; input cache อยู่ VM และอาจต้องสร้างใหม่เมื่อเปลี่ยน runtime
ทำซ้ำ cell เพื่อ resume; ถ้า config/code/dependency versions เปลี่ยน ใช้ output directory ใหม่


In [ ]:
from pathlib import Path
import importlib, importlib.util, sys, zipfile, shutil, subprocess

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/nih -xray/data/3")
    RELEASE_ROOT = Path("/content/drive/MyDrive/nih -xray/Notebook_3_release")
    CODE_ZIP = Path("/content/drive/MyDrive/nih -xray/Notebook_3_release_colab.zip")
    if CODE_ZIP.is_file():
        ROOT = Path("/content/Notebook_3_release")
        with zipfile.ZipFile(CODE_ZIP) as archive:
            for member in archive.infolist():
                relative = Path(member.filename)
                if relative.is_absolute() or ".." in relative.parts or not relative.parts or relative.parts[0] != "Notebook_3_release":
                    raise ValueError("Invalid source package path")
                if member.is_dir():
                    continue
                target = Path("/content").joinpath(*relative.parts).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError("Invalid source package destination")
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
    elif (RELEASE_ROOT / "pipeline.py").is_file():
        ROOT = RELEASE_ROOT
    else:
        raise FileNotFoundError("Upload Notebook_3_release_colab.zip to MyDrive/nih -xray, or upload the source folder as Notebook_3_release")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
    OUTPUT_ROOT = RELEASE_ROOT / "runs/release_full"
    CACHE_ROOT = Path("/content/cxr_v3_cache")
else:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Notebook_3 release"]
    ROOT = next((p for p in candidates if (p / "configs/release_full.json").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Open from Notebook_3 release or its repository directory")
    DATA_ROOT = None  # locate local data/versions/3 without changing the frozen manifests
    OUTPUT_ROOT = ROOT / "runs/release_full"
    CACHE_ROOT = ROOT / ".runtime/cache"

sys.path.insert(0, str(ROOT))
for name in tuple(sys.modules):
    if name == "shared" or name.startswith("shared.") or name in ("pipeline", "check_overfit", "export_results_colab"):
        del sys.modules[name]
importlib.invalidate_caches()
import torch
from pipeline import open_run, run_stages
SEEDS = [42, 43, 44]
SMOKE = False  # True = isolated tiny validation run with random initialization; not research scores
CONDITIONS = None  # None = all 12 controlled conditions
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Data:", DATA_ROOT, "| Results:", OUTPUT_ROOT, "| Device:", DEVICE)


In [ ]:
import threading
import time
from contextlib import contextmanager

@contextmanager
def preparation_progress(seed, interval=10):
    """Notebook-only observer; preserve all source files, pixels and run fingerprints."""
    import shared.data as data_module
    from shared.runtime_cache import RuntimeCache
    started = time.monotonic()
    status = {"phase": "opening run / checking configuration", "done": 0, "total": 0,
              "image": None, "reading_since": None}
    stop = threading.Event()
    original_manifests = data_module.prepare_manifests
    original_verify = data_module.verify_manifests
    original_prepare = RuntimeCache.prepare
    original_read = RuntimeCache._read

    def emit(message):
        print(f"[prepare seed={seed}] {message}", flush=True)

    def heartbeat():
        while not stop.wait(interval):
            elapsed = time.monotonic() - started
            message = f"{status['phase']} | elapsed {elapsed:.0f}s"
            if status["total"]:
                message += f" | {status['done']:,}/{status['total']:,} ({100*status['done']/status['total']:.1f}%)"
            if status["image"]:
                message += f" | image {status['image']}"
                if status["reading_since"] is not None:
                    message += f" | current read {time.monotonic()-status['reading_since']:.0f}s"
            emit(message)

    def manifests(exp):
        status["phase"] = "checking source checksums and loading frozen manifests"
        emit(status["phase"] + f" | data={exp.data_root}")
        result = original_manifests(exp)
        emit("manifests verified: " + str({name: len(frame) for name, frame in result.items()}))
        return result

    def verify(exp, frames):
        status["phase"] = "validating labels, patient partitions and bounding boxes"
        emit(status["phase"])
        result = original_verify(exp, frames)
        status["phase"] = "saving/verifying run manifest records"
        return result

    def prepare(cache, frames):
        status.update(phase="checking runtime cache and Drive folder layout", done=0,
                      total=len(set().union(*(set(frame.image) for frame in frames.values()))),
                      image=None, reading_since=None)
        emit(f"source cache: {status['total']:,} unique images | cache={cache.root}")
        result = original_prepare(cache, frames)
        status.update(phase="cache verified and ready", done=status["total"], image=None, reading_since=None)
        emit(f"cache ready: {status['total']:,} images (existing verified shards are reused)")
        return result

    def read(cache, row):
        status.update(phase="reading Drive image / resizing / caching", image=row["image"], reading_since=time.monotonic())
        if status["done"] == 0:
            emit(f"first image read: {row['image']}; heartbeat every {interval}s")
        result = original_read(cache, row)
        status["done"] = max(status["done"], cache.indices[row["image"]]+1)
        status["reading_since"] = None
        return result

    data_module.prepare_manifests = manifests
    data_module.verify_manifests = verify
    RuntimeCache.prepare = prepare
    RuntimeCache._read = read
    monitor = threading.Thread(target=heartbeat, daemon=True)
    emit("START: verifying data and preparing cache; model training has not started")
    monitor.start()
    try:
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        data_module.prepare_manifests = original_manifests
        data_module.verify_manifests = original_verify
        RuntimeCache.prepare = original_prepare
        RuntimeCache._read = original_read
        emit(f"preparation cell ended after {time.monotonic()-started:.1f}s")

runs = {}
for seed in SEEDS:
    with preparation_progress(seed):
        exp, manifests = open_run(DATA_ROOT, OUTPUT_ROOT, CACHE_ROOT, seed=seed, smoke=SMOKE)
    runs[seed] = (exp, manifests)
    print(seed, {name: len(frame) for name, frame in manifests.items()}, exp.output_root, flush=True)


In [ ]:
for seed, (exp, manifests) in runs.items():
    run_stages(exp, manifests, stages=["shap"], condition_ids=CONDITIONS, device=DEVICE)
